**Table of contents**<a id='toc0_'></a>    
- [Pivot Tables for the Paper](#toc1_)    
- [Let's Create Summary](#toc2_)    
- [Pivot Views](#toc3_)    
  - [Structural Text Elements](#toc3_1_)    
  - [Math](#toc3_2_)    
- [Categorical Groups](#toc4_)    
  - [Morphological](#toc4_1_)    
  - [LAnguage Contact](#toc4_2_)    
- [Orthography](#toc5_)    
  - [Input Medium](#toc5_1_)    
  - [Diacritics](#toc5_2_)    
  - [Register Style](#toc5_3_)    
  - [Morph](#toc5_4_)    
- [Noise Cuated](#toc6_)    
- [Grammar](#toc7_)    
- [Linguistic Variety](#toc8_)    
- [Structural Text](#toc9_)    
    - [Math styling](#toc9_1_1_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Pivot Tables for the Paper](#toc0_)



In [408]:
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator
import math
from pathlib import Path

from xarch_tokenizers.logging.report_utils import (
    load_predictions,
    load_all_samples,
    clean_model_name,
)
from xarch_tokenizers.logging.plot_utils import (
    setup_styles,
    get_new_6_fig,
    MODEL_TO_COLOR,
    get_new_fig,
)


setup_styles()
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [409]:
## same as exploratory plotting notebook `paper-benchmark-plots.ipynb`
def get_canonical_and_perturbed_df(
    samples,
    only_keep_canonical_true: bool = False,
    keep_remains_or_becomes_correct: bool = False,
    only_keep_perturbed_true: bool = False,
    verbose: bool = False,
):
    """returns filtered out samples too"""
    if (
        (only_keep_canonical_true and keep_remains_or_becomes_correct)
        or (only_keep_canonical_true and only_keep_perturbed_true)
        or (keep_remains_or_becomes_correct and only_keep_perturbed_true)
    ):
        raise ValueError(
            f"Please pass these exclusively, only_keep_canonical_true: true for canonical, model_name true, keep_remains_or_becomes_correct: True to include the ones where the perturbed version becomes correct"
        )
    if verbose:
        print(f"Processing {len(samples)} examples")
    canonical_samples = samples[samples["is_canonical"]]
    perturbed_samples = samples[~samples["is_canonical"]]
    canonical_values = canonical_samples[["set_id", "model_name", METRIC]].rename(
        columns={METRIC: f"canonical_{METRIC}"}
    )

    perturbed_samples = perturbed_samples.merge(
        canonical_values, on=["set_id", "model_name"], how="left"
    )
    perturbed_samples[f"canonical-perturbed_{METRIC}"] = (
        perturbed_samples[f"canonical_{METRIC}"] - perturbed_samples[METRIC]
    )

    if only_keep_canonical_true:
        correct_canonical_pairs = canonical_samples[canonical_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ]
        if verbose:
            print(
                f"Cleaning samples, will be dropping entries (set_id-model pairs) where the canonical is predicted wrong."
            )
            print(
                f"Keeping {len(correct_canonical_pairs)} correct pairs, dropping {len(canonical_samples) - len(correct_canonical_pairs)} entries."
            )

        canonical_samples = canonical_samples[canonical_samples[METRIC] == 1]

        # Step 3: Filter perturbed_samples to keep only rows with correct canonical pairs
        # Method 1: Using merge (RECOMMENDED)
        perturbed_samples = perturbed_samples.merge(
            correct_canonical_pairs,
            on=["set_id", "model_name"],
            how="inner",  # Only keep rows that exist in both
        )
        samples = pd.concat((canonical_samples, perturbed_samples), ignore_index=True)

    if keep_remains_or_becomes_correct:
        # Get pairs where perturbed samples are correct (becomes or remains correct)
        correct_perturbed_pairs = perturbed_samples[perturbed_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ].drop_duplicates()

        # Get pairs where canonical samples are correct (remains correct)
        correct_canonical_pairs = canonical_samples[canonical_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ]

        # Combine: pairs that either remain correct OR become correct
        remains_or_becomes_correct = pd.concat(
            [correct_canonical_pairs, correct_perturbed_pairs]
        ).drop_duplicates()

        # Filter both datasets to only include these pairs
        canonical_samples = canonical_samples.merge(
            remains_or_becomes_correct, on=["set_id", "model_name"], how="inner"
        )

        perturbed_samples = perturbed_samples.merge(
            remains_or_becomes_correct, on=["set_id", "model_name"], how="inner"
        )
        samples = pd.concat((canonical_samples, perturbed_samples), ignore_index=True)
        if verbose:
            print(f"After filtering:{len(samples)}")
    if only_keep_perturbed_true:
        # Get pairs where perturbed samples are correct (becomes or remains correct)
        correct_perturbed_pairs = perturbed_samples[perturbed_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ].drop_duplicates()

        # Get pairs where canonical samples are correct (remains correct)
        wrong_canonical_pairs = canonical_samples[canonical_samples[METRIC] == 0][
            ["set_id", "model_name"]
        ]
        # Find cases where canonical wrong AND perturbed correct (recovery cases)
        becomes_correct = pd.merge(
            wrong_canonical_pairs,
            correct_perturbed_pairs,
            how="inner",
            on=["set_id", "model_name"],
        )

        # Filter both datasets to only include recovery cases
        canonical_samples = canonical_samples.merge(
            becomes_correct, on=["set_id", "model_name"], how="inner"
        )

        perturbed_samples = perturbed_samples.merge(
            becomes_correct, on=["set_id", "model_name"], how="inner"
        )

        samples = pd.concat([canonical_samples, perturbed_samples], ignore_index=True)
        if verbose:
            print(f"After filtering: {len(samples)}")

    return samples, canonical_samples, perturbed_samples

# <a id='toc2_'></a>[Let's Create Summary](#toc0_)

In [410]:
OUTPUT_DIR = Path("./output/results-v5")
OUTPUT_DIR_ = OUTPUT_DIR / "summary"
METRIC = "acc_norm"
# read all samples
base_dir = Path("../results/paper-v5").resolve().absolute()

# # keeps set_id, model pairs that are correct for canonical
# suffix = "(Still correct)"
# only_keep_canonical_true = True
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = False

# suffix = "(Remains or Becomes Correct)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = True
# only_keep_perturbed_true = False

# suffix = "(Flipped to correct)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = True

# suffix = "(All)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = False

code_pattern = "*"
title_pattern = "All Datasets"
OUTPUT_DIR = OUTPUT_DIR_ / "all"

OUTPUT_DIR = OUTPUT_DIR
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
base_dir.exists()
pred_files = list([p for p in base_dir.rglob(f"samples{code_pattern}.jsonl")])
len(pred_files), pred_files[:5]


(2528,
 [PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_general_currency_symbol_2025-09-20T10-09-51.814046.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_farsi_canonical_2025-09-20T16-14-33.757080.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_chinese_ocr_errors_2025-09-20T11-23-04.491388.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_typographical_errors_2025-09-20T10-31-24.441216.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/s

In [434]:
## Read all samples into a DataFrame
## exclude general dataset for now
all_samples = load_all_samples(
    base_dir,
    patterns=[code_pattern],
    exclude_patterns=["general"],
    flatten_doc=True,
    match_date=False,
    simplify_df=True,
)

2366 [PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_farsi_canonical_2025-09-20T16-14-33.757080.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_chinese_ocr_errors_2025-09-20T11-23-04.491388.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_typographical_errors_2025-09-20T10-31-24.441216.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_grammatical_errors_2025-09-20T10-31-24.441216.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_t

We want a summary table like below

| filtering_mode   | model_name   | task                                                      | task_pretty_name     |   canonical_count |   num_samples | category                 | langs    |   acc_norm |   acc_norm_std |      acc |   acc_std | canonical_task_name                            |   canonical_acc |   canonical_acc_norm |
|:-----------------|:-------------|:----------------------------------------------------------|:---------------------|------------------:|--------------:|:-------------------------|:---------|-----------:|---------------:|---------:|----------:|:-----------------------------------------------|----------------:|---------------------:|
| no_filter        | Aya          | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.352941 |  0.492592 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | BLOOM        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | ByT5         | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Comma        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | GPT-2        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.235294 |  0.437237 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.823529 |
| no_filter        | GPT-4o       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.352941 |  0.492592 | tokenizer_robustness_completion_stem_canonical |        1        |             0.941176 |
| no_filter        | Gemma-2      | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.882353 |
| no_filter        | Llama-3.2    | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.882353 |
| no_filter        | Phi-3        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.235294 |  0.437237 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Qwen-3       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Tekken       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.235294 |       0.437237 | 0.176471 |  0.392953 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.882353 |
| no_filter        | TokenMonster | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.764706 |
| no_filter        | XGLM         | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.823529 |       0.392953 | 0.882353 |  0.332106 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | mBERT        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.235294 |       0.437237 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.764706 |


started with this

|          |                     |            |     |          |                   |                     |             |                  |                     |                     |
| -------- | ------------------- | ---------- | --- | -------- | ----------------- | ------------------- | ----------- | ---------------- | ------------------- | ------------------- |
| Category | Perturbation (Task) | Model name | acc | acc_norm | number of samples | filter              | acc_std_err | acc_norm_std_err | number of canonical | number of perturbed |
|          | romanization        | Comma      |     |          |                   | only_canonical_true |             |                  |                     |                     |
|          | dialect             | Comma      |     |          |                   | only_canonical_true |             |                  |                     |                     |
|          | romanization        | Comma      |     |          |                   | no_filter           |             |                  |                     |                     |
|          | dialect             | Comma      |     |          |                   | no_filter           |             |                  |                     |                     |
|          |                     |            |     |          |                   | only_canonical      |             |                  |                     |                     |
|          |                     |            |     |          |                   | only_perturbed      |             |                  |                     |                     |								

In [412]:
import warnings

from category_mapping import SUBCATEGORY_TO_CATEGORY
from typing import List

FILTERS = [
    "no_filter",
    "only_canonical_correct",
    "canonical_wrong_at_least_one_perturb_correct",
    "remains_or_becomes_correct",
]
keys = {
    "no_filter": (False, False, False),
    "only_canonical_correct": (True, False, False),
    "canonical_wrong_at_least_one_perturb_correct": (False, False, True),
    "remains_or_becomes_correct": (False, True, False),
}


def get_group_name(task_name):
    if "stem" in task_name:
        return "tokenizer_robustness_completion_stem"
    elif "english" in task_name:
        return "tokenizer_robustness_completion_english"
    elif "turkish" in task_name:
        return "tokenizer_robustness_completion_turkish"
    elif "farsi" in task_name:
        return "tokenizer_robustness_completion_farsi"
    elif "italian" in task_name:
        return "tokenizer_robustness_completion_italian"
    elif "chinese" in task_name:
        return "tokenizer_robustness_completion_chinese"
    elif "math" in task_name:
        return "tokenizer_robustness_completion_math"
    elif "general" in task_name:
        return "tokenizer_robustness_completion_general"


all_tasks = all_samples["task_pretty_name"].unique()
all_tasks = all_samples["task"].unique()
model_names = all_samples["model_name"].unique()
## cleanup secondary categories
all_samples["final_category"] = all_samples["category"].apply(
    lambda x: str(x).split(",")[0]
)
### TODO: replace later
# ## stem doesn't have lang for some reason, hacking it for now
# all_samples["lang"]=all_samples.apply(lambda row: "eng_Latn" if row["lang"] )


def get_summaries(subcategories: List[str] = None):
    all_summaries = []
    for filtering_mode in FILTERS:
        #################### filtering & clean-up ####################
        # don't rely on canonical_df, perturbed_df, always use samples
        samples, canonical_df, perturbed_df = get_canonical_and_perturbed_df(
            all_samples, *keys[filtering_mode], verbose=False
        )
        ## remove duplicate results just in case they were run twice
        dup_keys = [
            "model_name",
            "task",
            "subcategories",
            "lang",
            "set_id",
            "var_id",
            "question",
        ]
        duplicate_count = (
            samples.groupby(dup_keys, as_index=False).size()["size"] > 1
        ).sum()
        print(f"N duplicates: {duplicate_count}")
        samples = samples.drop_duplicates(subset=dup_keys)
        metadata = {"filtering_mode": filtering_mode}
        #################### process results for each model ####################
        for model in model_names:
            df_filter_ = samples["model_name"] == model
            metadata["model_name"] = model
            model_df = samples[df_filter_]
            #################### process results for each model & task pair ####################
            for task in all_tasks:
                tmp = model_df[model_df["task"] == task]
                if len(tmp) == 0 and filtering_mode == "no_filter":
                    warnings.warn(
                        f"Oh noo, check for model: {model}, task: {task} if it is run, the df is empty, skipping for now..."
                    )
                if len(tmp) == 0:
                    continue
                task_pretty_name = tmp["task_pretty_name"].unique()[0]
                if subcategories and task_pretty_name not in subcategories:
                    continue
                #################### extract canonical information ####################
                task_group_name = get_group_name(task)
                cor_canonical_task_name = f"{task_group_name}_canonical"
                ## apply filtering: canonical examples, match set ids and match tasks group name
                corr_canonical = model_df[
                    model_df["is_canonical"]
                    & model_df["set_id"].isin(tmp["set_id"].unique())
                    # & model_df[model_df["task"].str.startswith(task_group_name)]
                ]
                canonical_count = len(corr_canonical)

                if len(tmp) == 0:
                    pass
                    print(f"Empty df for task {task} under filter: {filtering_mode}")
                    continue
                category = tmp["final_category"].unique()
                category = SUBCATEGORY_TO_CATEGORY.get(task_pretty_name, "")
                metadata.update(
                    {
                        "task": task,
                        "task_pretty_name": task_pretty_name,
                        "canonical_count": canonical_count,
                        "num_samples": len(tmp),
                        "category": category,
                        "langs": ",".join([l for l in tmp["lang"].unique() if l != ""]),
                    }
                )
                # compute accuracy metrics
                acc_norm = tmp["acc_norm"].mean()
                acc_norm_std = tmp["acc_norm"].std()
                acc = tmp["acc"].mean()
                acc_std = tmp["acc"].std()
                summary = {
                    "acc_norm": tmp["acc_norm"].mean(),
                    "acc_norm_std": tmp["acc_norm"].std(),
                    "acc": tmp["acc"].mean(),
                    "acc_std": tmp["acc"].std(),
                    "canonical_task_name": cor_canonical_task_name,
                    "canonical_acc": corr_canonical["acc"].mean(),
                    "canonical_acc_norm": corr_canonical["acc_norm"].mean(),
                    "group_name": task_group_name,
                }
                all_summaries.append(metadata | summary)

    all_summaries = pd.DataFrame.from_records(all_summaries)
    return all_summaries


all_summaries = get_summaries()

N duplicates: 30604
N duplicates: 61193
N duplicates: 21964
N duplicates: 63314


In [435]:
## sanity check
print(
    all_summaries[
        all_summaries["task_pretty_name"] == "Fullwidth Characters"
    ].head(n=10).to_markdown(index=False)
)


| filtering_mode   | model_name   | task                                                      | task_pretty_name     |   canonical_count |   num_samples | category                 | langs    |   acc_norm |   acc_norm_std |      acc |   acc_std | canonical_task_name                            |   canonical_acc |   canonical_acc_norm | group_name                           |
|:-----------------|:-------------|:----------------------------------------------------------|:---------------------|------------------:|--------------:|:-------------------------|:---------|-----------:|---------------:|---------:|----------:|:-----------------------------------------------|----------------:|---------------------:|:-------------------------------------|
| no_filter        | Aya          | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.352941 |  0.492592 | toke

In [415]:
save_path = OUTPUT_DIR / "summary.tsv"

all_summaries.to_csv(save_path, sep="\t")
print(f"File saved at \n{save_path}")

File saved at 
output/results-v5/summary/all/summary.tsv


# <a id='toc3_'></a>[Pivot Views](#toc0_)

In [500]:
import pandas as pd
import numpy as np


# Modified function to exclude aggregation rows
def highlight_extremes(s):
    """
    Highlight the max value in green and min value in red for each column
    Excludes aggregation rows from min/max calculation
    """
    if s.dtype != "object":  # Only apply to numeric columns
        # Define patterns that identify aggregation rows
        aggregation_patterns = [
            "MEAN",
            "STD",
            "MIN",
            "MAX",
            "MEDIAN",
            "RANGE",
            "TOP_3_AVG",
            "BOTTOM_3_AVG",
            "TASK_WEIGHTED_AVG",
            "BEST_MODEL_PER_TASK",
            "WORST_MODEL_PER_TASK",
            "─",
            "═",
            "___",  # Visual separators
        ]

        # Filter out aggregation rows for min/max calculation
        model_only_series = s.copy()
        model_indices_to_exclude = []

        for idx in s.index:
            # Check if index matches any aggregation pattern
            if any(pattern in str(idx).upper() for pattern in aggregation_patterns):
                model_indices_to_exclude.append(idx)

        # Remove aggregation rows from calculation
        model_only_series = model_only_series.drop(
            model_indices_to_exclude, errors="ignore"
        )

        # Calculate min/max only from model rows
        if len(model_only_series) > 0:
            max_val = model_only_series.max()
            min_val = model_only_series.min()
        else:
            max_val = None
            min_val = None

        styles = []
        for idx, val in s.items():
            # Don't highlight aggregation rows at all
            if any(pattern in str(idx).upper() for pattern in aggregation_patterns):
                styles.append("")
            elif pd.isna(val):
                styles.append("")
            elif max_val is not None and val == max_val:
                styles.append(
                    "background-color: #90EE90; font-weight: bold; color: #006400"
                )  # Light green bg, dark green text
            elif min_val is not None and val == min_val:
                styles.append(
                    "background-color: #FFB6C1; font-weight: bold; color: #8B0000"
                )  # Light red bg, dark red text
            else:
                styles.append("")
        return styles
    else:
        return [""] * len(s)


def weighted_avg(group):
    """
    Calculate weighted average from a group that has both acc_norm and num_samples
    """
    if len(group) == 0:
        return "___"

    # group is a DataFrame with both columns
    acc_values = group["acc_norm"]
    weights = group["num_samples"]

    # Remove NaN values
    valid_mask = ~(pd.isna(acc_values) | pd.isna(weights)) & (weights > 0)

    if valid_mask.sum() > 0:
        return np.average(acc_values[valid_mask], weights=weights[valid_mask])
    else:
        return "___"


def get_styled_df(summaries, columns=["task_pretty_name", "langs"], index=["model_name"], ):
    pivot_df = (
        summaries.groupby([*index, *columns])
        .apply(weighted_avg)
        .reset_index()  # Convert back to DataFrame
        .rename(columns={0: "weighted_avg"})  # Name the result column
        .pivot_table(index=index, columns=columns, values="weighted_avg")
    )

    # Add aggregate columns
    pivot_df["Mean"] = pivot_df.mean(axis=1)
    # Sort by mean performance
    pivot_df = pivot_df.sort_values("Mean", ascending=False)

    # add column-wise stats
    pivot_df.loc["─" * 20] = np.nan  # Separator row
    pivot_df.loc["MEAN"] = pivot_df.mean(axis=0)
    pivot_df.loc["MAX"] = pivot_df.max(axis=0)
    pivot_df.loc["MIN"] = pivot_df.min(axis=0)
    pivot_df.loc["STD"] = pivot_df.std(axis=0)

    styled_df = (
        pivot_df.style.apply(highlight_extremes, axis=0)
        .format(precision=3)
        .set_table_styles(
            [
                {
                    "selector": "th.level0",
                    "props": [
                        ("border-right", "3px solid black"),
                        ("text-align", "center"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [("text-align", "center"), ("font-size", "11px")],
                },
            ]
        )
    )
    styled_df = styled_df.format(na_rep="─────")
    return styled_df

FILTER

In [501]:
FILTER

'only_canonical_correct'

In [502]:

style_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & ~(all_summaries["task"].str.contains("math"))
    & ~(all_summaries["task"].str.contains("stem"))
    # & all_summaries["task_pretty_name"].isin(styling_categories)
]
styled_df = get_styled_df(style_summaries, columns=["task_pretty_name", "langs"])
# styled_df = get_styled_df(style_summaries, index=["task_pretty_name", "langs"], columns=["model_name"])
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [503]:
styled_df

## Main Take-aways

In [ ]:
all_summaries[all_summaries["task_pretty_name"].isin(nfd_affected_styles)][""]

,filtering_mode,model_name,task,task_pretty_name,canonical_count,num_samples,category,langs,acc_norm,acc_norm_std,acc,acc_std,canonical_task_name,canonical_acc,canonical_acc_norm,group_name,language_split
3,no_filter,Aya,tokenizer_robustness_completion_math_decorativ...,Decorative Unicode,21,21,Mathematical & Scientific Notation,eng_Latn,0.428571,0.507093,0.333333,0.483046,tokenizer_robustness_completion_math_canonical,0.619048,0.666667,tokenizer_robustness_completion_math,EN
22,no_filter,Aya,tokenizer_robustness_completion_chinese_option...,Optional diacritics,160,40,Script / Orthography,zho_Hans,0.375000,0.490290,0.200000,0.405096,tokenizer_robustness_completion_chinese_canonical,0.818750,0.831250,tokenizer_robustness_completion_chinese,Non-EN
33,no_filter,Aya,tokenizer_robustness_completion_farsi_optional...,Optional diacritics,160,40,Script / Orthography,pes_Arab,0.425000,0.500641,0.450000,0.503831,tokenizer_robustness_completion_farsi_canonical,0.818750,0.831250,tokenizer_robustness_completion_farsi,Non-EN
69,no_filter,Aya,tokenizer_robustness_completion_english_macron...,Macron Diacritic,160,40,Structural Text Elements,eng_Latn,0.250000,0.438529,0.225000,0.422902,tokenizer_robustness_completion_english_canonical,0.818750,0.831250,tokenizer_robustness_completion_english,EN
71,no_filter,Aya,tokenizer_robustness_completion_english_script...,Scripted text,160,40,Structural Text Elements,eng_Latn,0.325000,0.474342,0.200000,0.405096,tokenizer_robustness_completion_english_canonical,0.818750,0.831250,tokenizer_robustness_completion_english,EN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6416,remains_or_becomes_correct,mBERT,tokenizer_robustness_completion_stem_diacritic...,Diacriticized styling,13,55,Structural Text Elements,eng_Latn,0.272727,0.449467,0.254545,0.439620,tokenizer_robustness_completion_stem_canonical,1.000000,0.846154,tokenizer_robustness_completion_stem,EN
6418,remains_or_becomes_correct,mBERT,tokenizer_robustness_completion_stem_double_st...,Double struck,17,17,Structural Text Elements,eng_Latn,0.294118,0.469668,0.235294,0.437237,tokenizer_robustness_completion_stem_canonical,0.882353,0.823529,tokenizer_robustness_completion_stem,EN
6419,remains_or_becomes_correct,mBERT,tokenizer_robustness_completion_stem_enclosed_...,Enclosed Characters,13,42,Structural Text Elements,eng_Latn,0.166667,0.377195,0.119048,0.327770,tokenizer_robustness_completion_stem_canonical,1.000000,0.846154,tokenizer_robustness_completion_stem,EN
6420,remains_or_becomes_correct,mBERT,tokenizer_robustness_completion_stem_fullwidth...,Fullwidth Characters,16,16,Structural Text Elements,eng_Latn,0.250000,0.447214,0.312500,0.478714,tokenizer_robustness_completion_stem_canonical,0.875000,0.812500,tokenizer_robustness_completion_stem,EN


In [657]:
all_summaries["language_split"] = all_summaries["langs"].apply(lambda x: "EN" if x == "eng_Latn" else "Non-EN")
writing_system_categories = [
    # "Orthographic errors", 
    # "Phonetic spelling", 
    "English keyboard", 
    "Arabic Keyboard for Farsi",
    "Number Romanization"
]
orthographic_errors = [
    # "Homoglyphs",
    "Orthographic errors"
]

noise_categories = [
    "Homoglyphs",
    "Plausible diacritics errors", 
    "Keyboard proximity errors", 
    "OCR Errors", 
    "Character deletion", 
    "Space removal", 
    "Typographical errors"
    "Word Spacing/Zero-width characters/Extra Space", 
]

nfd_affected_styles = [
    # "Optional diacritics", 
    # "Macron Diacritic", 
    # "Diacriticized styling", 
    "Decorative Unicode", 
    "Fullwidth Characters", 
    "Scripted text", 
    "Double struck", 
    "Enclosed Characters", 
    "Unicode formatting",
    "Superscript/subscript",
]

other_style_categories = [
    # "Capitalization", 
    "Lowercase", 
    "Strikethrough", 
    "Upside Down/Rotated", 
    "Spaced styling", 
    "Hyphenated spelling"
]

errors_that_change_token_boundaries = [
    # "Orthographic errors", 
    "Space removal", 
    "Character deletion", 
    "Hyphenated spelling", 
    "Spaced styling"
] #+ noise_categories

math_and_stem_categories = [
    # "LaTeX", 
    "Spelled out", 
    "Unusual formatting", 
    # "Numerical formats", 
    # # "Superscript/subscript styling", 
    # # "Superscript/subscript", 
    "Turkish", "Italian", "Chinese", "Farsi"
]

socials = [
    "Colloquial", 
    "Emoji substitution", 
    "Character substitution", 
    "Letter repetition for emphasis", 
    "Web search query", 
    "Abbreviations"
]

In [658]:
all_summaries["task_pretty_name"].unique()

array(['Canonical', 'LaTeX', 'Spelled out', 'Decorative Unicode', 'Farsi',
       'Turkish', 'Italian', 'Chinese', 'Space removal',
       'Orthographic errors', 'Plausible diacritics errors',
       'English keyboard', 'Typographical errors', 'Grammatical errors',
       'Contractions', 'Code/language/script switching', 'Capitalization',
       'Optional diacritics', 'Colloquial', 'Dialects', 'Compounds',
       'Inflections', 'Emoji substitution', 'Abbreviations', 'OCR Errors',
       'Keyboard proximity errors',
       'Word Spacing/Zero-width characters/Extra Space',
       'Word reordering', 'Romanization', 'Traditional', 'Derivations',
       'Equivalent expressions', 'Character deletion',
       'Partially romanized', 'Arabic Keyboard for Farsi',
       'Historical spelling', 'Homoglyphs', 'Hyphenated spelling',
       'Similar words', 'Letter repetition for emphasis',
       'Web search query', 'Lowercase', 'Macron Diacritic',
       'Scripted text', 'Spaced styling', 'Superscr

In [659]:
# Define category mappings
category_mapping = {
    "Writing Systems": writing_system_categories,
    "Orthographic Errors": orthographic_errors, 
    "Noise Categories": noise_categories,
    "NFD Affected Styles": nfd_affected_styles,
    # "Other Style Categories": other_style_categories,
    "Token Boundary Errors": errors_that_change_token_boundaries,
    "LaTeX": ["LaTeX"],
    "Math & STEM": math_and_stem_categories,
    "Register & Style": socials
}

# Create category performance summary with hierarchical columns
def create_hierarchical_category_summary(all_summaries, category_mapping, filter_mode="only_canonical_correct"):
    """
    Create a summary table with hierarchical columns: Category -> Language
    """
    # Filter data
    filtered_summaries = all_summaries[all_summaries["filtering_mode"] == filter_mode].copy()
    
    # Create hierarchical structure
    results_data = []
    
    for model in filtered_summaries["model_name"].unique():
        model_data = filtered_summaries[filtered_summaries["model_name"] == model]
        row_data = []
        
        for category_name, perturbations in category_mapping.items():
            category_data = model_data[model_data["task_pretty_name"].isin(perturbations)]
            
            # English performance
            eng_data = category_data[category_data["langs"] == "eng_Latn"]
            if len(eng_data) > 0:
                acc_values = eng_data["acc_norm"]
                weights = eng_data["num_samples"]
                valid_mask = ~(pd.isna(acc_values) | pd.isna(weights)) & (weights > 0)
                if valid_mask.sum() > 0:
                    eng_avg = np.average(acc_values[valid_mask], weights=weights[valid_mask])
                else:
                    eng_avg = np.nan
            else:
                eng_avg = np.nan
            
            # Non-English performance
            non_eng_data = category_data[category_data["langs"] != "eng_Latn"]
            if len(non_eng_data) > 0:
                acc_values = non_eng_data["acc_norm"]
                weights = non_eng_data["num_samples"]
                valid_mask = ~(pd.isna(acc_values) | pd.isna(weights)) & (weights > 0)
                if valid_mask.sum() > 0:
                    non_eng_avg = np.average(acc_values[valid_mask], weights=weights[valid_mask])
                else:
                    non_eng_avg = np.nan
            else:
                non_eng_avg = np.nan
            
            row_data.extend([eng_avg, non_eng_avg])
        
        results_data.append([model] + row_data)
    
    # Create column structure
    columns = ['model_name']
    column_tuples = []
    
    for category_name in category_mapping.keys():
        column_tuples.extend([
            (category_name, 'English'),
            (category_name, 'Non-English')
        ])
    
    # Create DataFrame
    df = pd.DataFrame(results_data, columns=columns + [f"{cat}_{lang}" for cat, lang in column_tuples])
    
    # Create MultiIndex columns
    multi_columns = pd.MultiIndex.from_tuples([('Model', '')] + column_tuples, names=['Category', 'Language'])
    
    # Reconstruct with proper MultiIndex
    df_multi = pd.DataFrame(results_data, columns=['model_name'] + [f"{cat}_{lang}" for cat, lang in column_tuples])
    df_multi = df_multi.set_index('model_name')
    df_multi.columns = pd.MultiIndex.from_tuples(column_tuples, names=['Category', 'Language'])
    
    return df_multi


# Create hierarchical styled table function
def get_hierarchical_styled_df(summary_df):
    """
    Create styled dataframe for hierarchical category summary
    """
    styled_data = summary_df.copy()
    
    # Add overall statistics for each category (across languages)
    for category in styled_data.columns.get_level_values(0).unique():
        category_cols = styled_data[category]
        styled_data[(category, 'Mean')] = category_cols.mean(axis=1)
    
    # Add overall mean across all categories and languages
    all_numeric_cols = styled_data.select_dtypes(include=[np.number])
    styled_data[('Overall', 'Mean')] = all_numeric_cols.mean(axis=1)
    
    # Sort by overall performance
    styled_data = styled_data.sort_values(('Overall', 'Mean'), ascending=False)
    
    # Add summary statistics rows
    styled_data.loc["─" * 20] = np.nan  # Separator row
    styled_data.loc["MEAN"] = styled_data.mean(axis=0)
    styled_data.loc["MAX"] = styled_data.max(axis=0) 
    styled_data.loc["MIN"] = styled_data.min(axis=0)
    styled_data.loc["STD"] = styled_data.std(axis=0)
    styled_data.loc["RANGE"] = styled_data.max(axis=0) - styled_data.min(axis=0)
    
    # Apply styling
    styled_df = (
        styled_data.style.apply(highlight_extremes, axis=0)
        .format(precision=3)
        .set_table_styles([
            {
                "selector": "th.level0",
                "props": [
                    ("text-align", "center"),
                    ("font-weight", "bold"),
                    ("background-color", "#f0f0f0"),
                    ("border-right", "2px solid black")
                ]
            },
            {
                "selector": "th.level1", 
                "props": [
                    ("text-align", "center"),
                    ("font-weight", "normal"),
                    ("background-color", "#f8f8f8")
                ]
            },
            {
                "selector": "td", 
                "props": [
                    ("text-align", "center"), 
                    ("font-size", "11px")
                ]
            }
        ])
    )
    
    styled_df = styled_df.format(na_rep="─────")
    return styled_df


# Create the hierarchical category summary
FILTER = "only_canonical_correct"
hierarchical_summary = create_hierarchical_category_summary(all_summaries, category_mapping, FILTER)

# Generate the styled table
print("=== HIERARCHICAL CATEGORY SUMMARY (Category -> Language) ===")
styled_hierarchical_df = get_hierarchical_styled_df(hierarchical_summary)
styled_hierarchical_df = styled_hierarchical_df.set_caption(f"Hierarchical Category Performance: Category -> Language ({FILTER})")
styled_hierarchical_df




=== HIERARCHICAL CATEGORY SUMMARY (Category -> Language) ===


In [664]:
import pandas as pd
import numpy as np

# Define the specific categories for your LaTeX table
latex_table_categories = {
    "Writing Systems (Non-EN)": [
        "English keyboard", 
        "Arabic Keyboard for Farsi",
        "Number Romanization",
        "Romanization"
    ],
    "Noise (EN)": [
        "Homoglyphs",
        "Plausible diacritics errors", 
        "Keyboard proximity errors", 
        "OCR Errors", 
        "Character deletion", 
        "Space removal", 
        "Typographical errors",
        "Word Spacing/Zero-width characters/Extra Space"
    ],
    "Noise (Non-EN)": [
        "Homoglyphs",
        "Plausible diacritics errors", 
        "Keyboard proximity errors", 
        "OCR Errors", 
        "Character deletion", 
        "Space removal", 
        "Typographical errors",
        "Word Spacing/Zero-width characters/Extra Space"
    ],
    "Whitespace": [
        "Space removal", 
        "Word Spacing/Zero-width characters/Extra Space", 
        "Hyphenated spelling", 
        "Spaced styling"
    ],
    "LaTeX": [
        "LaTeX",
    ],
    "Math/STEM (EN)": [
        "Spelled out", 
        "Unusual formatting"
    ],
    "Math/STEM (Non-EN)": [
        "Turkish", 
        "Italian", 
        "Chinese", 
        "Farsi"
    ],
    "NFD Affected": [
        # "Macron Diacritic", 
        # "Diacriticized styling"
    "Decorative Unicode", 
    "Fullwidth Characters", 
    "Scripted text", 
    "Double struck", 
    "Enclosed Characters", 
    "Unicode formatting",
    "Superscript/subscript",
    ],
    "Diacritics (Non-EN)": [
        "Optional diacritics",
        # "Macron Diacritic", 
        # "Diacriticized styling",
        # "Plausible diacritics errors"
    ]
}

def extract_latex_table_data(all_summaries, filter_mode="only_canonical_correct"):
    """
    Extract data for LaTeX table with specific column structure
    """
    # Filter data
    filtered_summaries = all_summaries[all_summaries["filtering_mode"] == filter_mode].copy()
    
    # Add language split
    filtered_summaries["language_split"] = filtered_summaries["langs"].apply(
        lambda x: "EN" if x == "eng_Latn" else "Non-EN"
    )
    
    results = {}
    
    for model in filtered_summaries["model_name"].unique():
        model_data = filtered_summaries[filtered_summaries["model_name"] == model]
        
        model_results = {"model_name": model}
        
        # 1. Canonical baseline (before any filtering)
        canonical_data = model_data[model_data["task_pretty_name"] == "Canonical"]
        if len(canonical_data) > 0:
            acc_values = canonical_data["acc_norm"]
            weights = canonical_data["num_samples"]
            valid_mask = ~(pd.isna(acc_values) | pd.isna(weights)) & (weights > 0)
            if valid_mask.sum() > 0:
                canonical_avg = np.average(acc_values[valid_mask], weights=weights[valid_mask])
                model_results["Canonical"] = canonical_avg
            else:
                model_results["Canonical"] = np.nan
        else:
            model_results["Canonical"] = np.nan
        
        # 2. Writing Systems (Non-EN only)
        writing_data = model_data[
            (model_data["task_pretty_name"].isin(latex_table_categories["Writing Systems (Non-EN)"])) &
            (model_data["language_split"] == "Non-EN")
        ]
        model_results["Writing Systems (Non-EN)"] = calculate_weighted_avg(writing_data)
        
        # 3. Noise (EN and Non-EN separately)
        for lang in ["EN", "Non-EN"]:
            noise_data = model_data[
                (model_data["task_pretty_name"].isin(latex_table_categories[f"Noise ({lang})"])) &
                (model_data["language_split"] == lang)
            ]
            model_results[f"Noise ({lang})"] = calculate_weighted_avg(noise_data)
        
        # 4. Whitespace (combined EN + Non-EN)
        whitespace_data = model_data[
            model_data["task_pretty_name"].isin(latex_table_categories["Whitespace"])
        ]
        model_results["Whitespace"] = calculate_weighted_avg(whitespace_data)
        
        # 5. LaTeX (combined EN + Non-EN)
        latex_data = model_data[
            model_data["task_pretty_name"].isin(latex_table_categories["LaTeX"])
        ]
        model_results["LaTeX"] = calculate_weighted_avg(latex_data)
        
        # 6. Math/STEM (Non-EN only)
        stem_data = model_data[
            (model_data["task_pretty_name"].isin(latex_table_categories["Math/STEM (Non-EN)"])) &
            (model_data["language_split"] == "Non-EN")
        ]
        model_results["Math/STEM (Non-EN)"] = calculate_weighted_avg(stem_data)
        
        # 7. Structural (combined EN + Non-EN)
        structural_data = model_data[
            model_data["task_pretty_name"].isin(latex_table_categories["NFD Affected"])
        ]
        model_results["NFD Affected"] = calculate_weighted_avg(structural_data)
        
        # 8. Diacritics (Non-EN only)
        diacritics_data = model_data[
            (model_data["task_pretty_name"].isin(latex_table_categories["Diacritics (Non-EN)"])) &
            (model_data["language_split"] == "Non-EN")
        ]
        model_results["Diacritics (Non-EN)"] = calculate_weighted_avg(diacritics_data)
        
        results[model] = model_results
    
    return results

def calculate_weighted_avg(data):
    """Calculate weighted average from filtered data"""
    if len(data) == 0:
        return np.nan
    
    acc_values = data["acc_norm"]
    weights = data["num_samples"]
    valid_mask = ~(pd.isna(acc_values) | pd.isna(weights)) & (weights > 0)
    
    if valid_mask.sum() > 0:
        return np.average(acc_values[valid_mask], weights=weights[valid_mask])
    else:
        return np.nan

def results_to_dataframe(results_dict):
    """Convert results dictionary to DataFrame"""
    df = pd.DataFrame.from_dict(results_dict, orient='index')
    df = df.drop('model_name', axis=1)  # Remove redundant column
    return df

def format_for_latex(df, precision=3):
    """Format DataFrame for LaTeX output"""
    # Round to specified precision
    df_formatted = df.round(precision)
    
    # Replace NaN with dashes
    df_formatted = df_formatted.fillna('---')
    
    # Sort by canonical performance
    if 'Canonical' in df_formatted.columns:
        df_formatted = df_formatted.sort_values('Canonical', ascending=False)
    
    return df_formatted

def generate_latex_table_dict(all_summaries, filter_mode="only_canonical_correct"):
    """Main function to generate the complete data structure"""
    
    # Extract raw data
    results = extract_latex_table_data(all_summaries, filter_mode)
    
    # Convert to DataFrame
    df = results_to_dataframe(results)
    
    # Format for LaTeX
    df_latex = format_for_latex(df)
    
    # Create column order as specified
    column_order = [
        # "Canonical",
        "Writing Systems (Non-EN)", 
        "Noise (EN)",
        "Noise (Non-EN)",
        "Whitespace",
        "LaTeX",
        "Math/STEM (Non-EN)",
        "NFD Affected", 
        "Diacritics (Non-EN)"
    ]
    
    # Reorder columns
    df_latex = df_latex[column_order]
    
    # Convert to dictionary format for easy LaTeX generation
    latex_dict = {
        "data": df_latex.to_dict('index'),
        "columns": column_order,
        "models": df_latex.index.tolist(),
        "raw_dataframe": df_latex
    }
    
    return latex_dict

# Generate the data
FILTER = "only_canonical_correct"
latex_data = generate_latex_table_dict(all_summaries, FILTER)

# Display the structured data
print("LaTeX Table Data Structure:")
print("="*50)
print(f"Models: {len(latex_data['models'])}")
print(f"Columns: {latex_data['columns']}")
print("\nSample data:")
for i, (model, data) in enumerate(list(latex_data['data'].items())[:3]):
    print(f"{model}: {data}")
    
print(f"\nDataFrame shape: {latex_data['raw_dataframe'].shape}")
print("\nDataFrame preview:")
print(latex_data['raw_dataframe'].head())

# Optional: Generate LaTeX table string
def generate_latex_table_string(latex_dict, caption="Multilingual Tokenization Performance", label="tab:multilingual_performance"):
    """Generate complete LaTeX table string"""
    
    df = latex_dict['raw_dataframe']
    
    # Create column specification
    n_cols = len(df.columns)
    col_spec = "l" + "c" * n_cols  # left-aligned for model names, centered for data
    
    # Start table
    latex_str = f"""\\begin{{table}}[ht]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
\\begin{{tabular}}{{{col_spec}}}
\\toprule
"""
    
    # Header row
    header = "Model & " + " & ".join([col.replace("(", "\\\\(").replace(")", "\\\\)") for col in df.columns]) + " \\\\\n"
    latex_str += header
    latex_str += "\\midrule\n"
    
    # Data rows
    for model, row in df.iterrows():
        model_clean = model.replace("_", "\\_")  # Escape underscores
        row_str = model_clean + " & " + " & ".join([str(val) for val in row.values]) + " \\\\\n"
        latex_str += row_str
    
    # End table
    latex_str += """\\bottomrule
\\end{tabular}
\\end{table}"""
    
    return latex_str

# Generate LaTeX string
latex_table_string = generate_latex_table_string(latex_data)
print("\n" + "="*50)
print("LaTeX Table String:")
print("="*50)
print(latex_table_string)

LaTeX Table Data Structure:
Models: 14
Columns: ['Writing Systems (Non-EN)', 'Noise (EN)', 'Noise (Non-EN)', 'Whitespace', 'LaTeX', 'Math/STEM (Non-EN)', 'NFD Affected', 'Diacritics (Non-EN)']

Sample data:
Aya: {'Writing Systems (Non-EN)': 0.567, 'Noise (EN)': 0.799, 'Noise (Non-EN)': 0.673, 'Whitespace': 0.59, 'LaTeX': 0.675, 'Math/STEM (Non-EN)': 0.732, 'NFD Affected': 0.404, 'Diacritics (Non-EN)': 0.4}
BLOOM: {'Writing Systems (Non-EN)': 0.543, 'Noise (EN)': 0.816, 'Noise (Non-EN)': 0.707, 'Whitespace': 0.557, 'LaTeX': 0.779, 'Math/STEM (Non-EN)': 0.603, 'NFD Affected': 0.414, 'Diacritics (Non-EN)': 0.462}
ByT5: {'Writing Systems (Non-EN)': 0.559, 'Noise (EN)': 0.823, 'Noise (Non-EN)': 0.716, 'Whitespace': 0.568, 'LaTeX': 0.759, 'Math/STEM (Non-EN)': 0.717, 'NFD Affected': 0.443, 'Diacritics (Non-EN)': 0.423}

DataFrame shape: (14, 8)

DataFrame preview:
       Writing Systems (Non-EN)  Noise (EN)  Noise (Non-EN)  Whitespace  \
Aya                       0.567       0.799           

In [679]:
import pandas as pd
import numpy as np

# Define the specific categories for your LaTeX table
latex_table_categories = {
    "Writing Systems (Non-EN)": [
        "English keyboard", 
        "Arabic Keyboard for Farsi",
        "Number Romanization"
    ],
    "Noise (EN)": [
        "Homoglyphs",
        "Plausible diacritics errors", 
        "Keyboard proximity errors", 
        "OCR Errors", 
        "Character deletion", 
        "Space removal", 
        "Typographical errors",
        "Word Spacing/Zero-width characters/Extra Space"
    ],
    "Noise (Non-EN)": [
        "Homoglyphs",
        "Plausible diacritics errors", 
        "Keyboard proximity errors", 
        "OCR Errors", 
        "Character deletion", 
        "Space removal", 
        "Typographical errors",
        "Word Spacing/Zero-width characters/Extra Space"
    ],
    "Whitespace": [
        "Space removal", 
        "Word Spacing/Zero-width characters/Extra Space", 
        "Hyphenated spelling", 
        "Spaced styling"
    ],
    "LaTeX": [
        "LaTeX",
        "Spelled out", 
        "Unusual formatting"
    ],
    "Math/STEM (Non-EN)": [
        "Turkish", 
        "Italian", 
        "Chinese", 
        "Farsi"
    ],
    "Structural": [
        "Macron Diacritic", 
        "Diacriticized styling", 
        "Fullwidth Characters", 
        "Decorative Unicode", 
        "Scripted text", 
        "Double struck", 
        "Enclosed Characters", 
        "Unicode formatting",
        "Lowercase", 
        "Strikethrough", 
        "Upside Down/Rotated"
    ],
    "Diacritics (Non-EN)": [
        "Optional diacritics",
        "Macron Diacritic", 
        "Diacriticized styling",
        "Plausible diacritics errors"
    ]
}

def extract_latex_table_data(all_summaries, filter_mode="only_canonical_correct"):
    """
    Extract data for LaTeX table with specific column structure
    """
    # Filter data
    filtered_summaries = all_summaries[all_summaries["filtering_mode"] == filter_mode].copy()
    
    # Add language split
    filtered_summaries["language_split"] = filtered_summaries["langs"].apply(
        lambda x: "EN" if x == "eng_Latn" else "Non-EN"
    )
    
    results = {}
    
    for model in filtered_summaries["model_name"].unique():
        model_data = filtered_summaries[filtered_summaries["model_name"] == model]
        
        model_results = {"model_name": model}
        
        # 1. Canonical baseline (before any filtering)
        canonical_data = model_data[model_data["task_pretty_name"] == "Canonical"]
        if len(canonical_data) > 0:
            acc_values = canonical_data["acc_norm"]
            weights = canonical_data["num_samples"]
            valid_mask = ~(pd.isna(acc_values) | pd.isna(weights)) & (weights > 0)
            if valid_mask.sum() > 0:
                canonical_avg = np.average(acc_values[valid_mask], weights=weights[valid_mask])
                model_results["Canonical"] = canonical_avg
            else:
                model_results["Canonical"] = np.nan
        else:
            model_results["Canonical"] = np.nan
        
        # 2. Writing Systems (Non-EN only)
        writing_data = model_data[
            (model_data["task_pretty_name"].isin(latex_table_categories["Writing Systems (Non-EN)"])) &
            (model_data["language_split"] == "Non-EN")
        ]
        model_results["Writing Systems (Non-EN)"] = calculate_weighted_avg(writing_data)
        
        # 3. Noise (EN and Non-EN separately)
        for lang in ["EN", "Non-EN"]:
            noise_data = model_data[
                (model_data["task_pretty_name"].isin(latex_table_categories[f"Noise ({lang})"])) &
                (model_data["language_split"] == lang)
            ]
            model_results[f"Noise ({lang})"] = calculate_weighted_avg(noise_data)
        
        # 4. Whitespace (combined EN + Non-EN)
        whitespace_data = model_data[
            model_data["task_pretty_name"].isin(latex_table_categories["Whitespace"])
        ]
        model_results["Whitespace"] = calculate_weighted_avg(whitespace_data)
        
        # 5. LaTeX (combined EN + Non-EN)
        latex_data = model_data[
            model_data["task_pretty_name"].isin(latex_table_categories["LaTeX"])
        ]
        model_results["LaTeX"] = calculate_weighted_avg(latex_data)
        
        # 6. Math/STEM (Non-EN only)
        stem_data = model_data[
            (model_data["task_pretty_name"].isin(latex_table_categories["Math/STEM (Non-EN)"])) &
            (model_data["language_split"] == "Non-EN")
        ]
        model_results["Math/STEM (Non-EN)"] = calculate_weighted_avg(stem_data)
        
        # 7. Structural (combined EN + Non-EN)
        structural_data = model_data[
            model_data["task_pretty_name"].isin(latex_table_categories["Structural"])
        ]
        model_results["Structural"] = calculate_weighted_avg(structural_data)
        
        # 8. Diacritics (Non-EN only)
        diacritics_data = model_data[
            (model_data["task_pretty_name"].isin(latex_table_categories["Diacritics (Non-EN)"])) &
            (model_data["language_split"] == "Non-EN")
        ]
        model_results["Diacritics (Non-EN)"] = calculate_weighted_avg(diacritics_data)
        
        results[model] = model_results
    
    return results

def calculate_weighted_avg(data):
    """Calculate weighted average from filtered data"""
    if len(data) == 0:
        return np.nan
    
    acc_values = data["acc_norm"]
    weights = data["num_samples"]
    valid_mask = ~(pd.isna(acc_values) | pd.isna(weights)) & (weights > 0)
    
    if valid_mask.sum() > 0:
        return np.average(acc_values[valid_mask], weights=weights[valid_mask])
    else:
        return np.nan

def results_to_dataframe(results_dict):
    """Convert results dictionary to DataFrame"""
    df = pd.DataFrame.from_dict(results_dict, orient='index')
    df = df.drop('model_name', axis=1)  # Remove redundant column
    return df

def format_for_latex(df, precision=3):
    """Format DataFrame for LaTeX output"""
    # Round to specified precision
    df_formatted = df.round(precision)
    
    # Replace NaN with dashes
    df_formatted = df_formatted.fillna('---')
    
    # Sort by canonical performance
    if 'Canonical' in df_formatted.columns:
        df_formatted = df_formatted.sort_values('Canonical', ascending=False)
    
    return df_formatted

# Generate the data
FILTER = "only_canonical_correct"
latex_data = generate_latex_table_dict(all_summaries, FILTER)

# Display the structured data
print("LaTeX Table Data Structure:")
print("="*50)
print(f"Models: {len(latex_data['models'])}")
print(f"Columns: {latex_data['columns']}")
print("\nSample data:")
for i, (model, data) in enumerate(list(latex_data['data'].items())[:3]):
    print(f"{model}: {data}")
    
print(f"\nDataFrame shape: {latex_data['raw_dataframe'].shape}")
print("\nDataFrame preview:")
print(latex_data['raw_dataframe'].head())

# Optional: Generate LaTeX table string
def generate_latex_table_string(latex_dict, caption="Multilingual Tokenization Performance", label="tab:multilingual_performance"):
    """Generate complete LaTeX table string"""
    
    df = latex_dict['raw_dataframe']
    
    # Create column specification
    n_cols = len(df.columns)
    col_spec = "l" + "c" * n_cols  # left-aligned for model names, centered for data
    
    # Start table
    latex_str = f"""\\begin{{table}}[ht]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
\\begin{{tabular}}{{{col_spec}}}
\\toprule
"""
    
    # Header row
    header = "Model & " + " & ".join([col.replace("(", "\\\\(").replace(")", "\\\\)") for col in df.columns]) + " \\\\\n"
    latex_str += header
    latex_str += "\\midrule\n"
    
    # Data rows
    for model, row in df.iterrows():
        model_clean = model.replace("_", "\\_")  # Escape underscores
        row_str = model_clean + " & " + " & ".join([str(val) for val in row.values]) + " \\\\\n"
        latex_str += row_str
    
    # End table
    latex_str += """\\bottomrule
\\end{tabular}
\\end{table}"""
    
    return latex_str

# Save results to JSONL file
import json

def save_to_jsonl(latex_dict, filename="tokenization_results.jsonl"):
    """Save results to JSONL format for later processing"""
    
    # Convert DataFrame to records format
    records = []
    for model_name, row in latex_dict['raw_dataframe'].iterrows():
        record = {"model_name": model_name}
        for col, value in row.items():
            record[col] = value if not pd.isna(value) else None
        records.append(record)
    
    # Write to JSONL
    with open(filename, 'w') as f:
        for record in records:
            f.write(json.dumps(record) + '\n')
    
    print(f"Results saved to {filename}")
    return filename

# Save data
jsonl_file = save_to_jsonl(latex_data)

# Function to load JSONL and create LaTeX table with formatting
def load_jsonl_and_create_latex(filename, whitespace_data=None, caption="Multilingual Tokenization Performance", label="tab:multilingual_performance"):
    """
    Load JSONL data and create formatted LaTeX table with best/worst highlighting
    whitespace_data: optional dict with model_name -> whitespace_score mapping
    """
    
    # Load data
    records = []
    with open(filename, 'r') as f:
        for line in f:
            records.append(json.loads(line))
    
    # Convert to DataFrame
    df = pd.DataFrame(records).set_index('model_name')
    
    # Add whitespace data if provided
    if whitespace_data:
        df['Whitespace'] = df.index.map(whitespace_data)
    
    # Sort by average performance across all columns
    df['_avg_score'] = df.mean(axis=1, skipna=True)
    df = df.sort_values('_avg_score', ascending=False)
    df = df.drop('_avg_score', axis=1)
    
    # Create column specification
    n_cols = len(df.columns)
    col_spec = "l" + "c" * n_cols
    
    # Start table
    latex_str = f"""\\begin{{table}}[ht]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
\\begin{{tabular}}{{{col_spec}}}
\\toprule
"""
    
    # Create header with proper escaping
    header_names = []
    for col in df.columns:
        col_clean = col.replace("(", "\\textsubscript{").replace(")", "}")
        col_clean = col_clean.replace("Non-EN", "Non-EN").replace("EN", "EN")
        header_names.append(col_clean)
    
    header = "Model & " + " & ".join(header_names) + " \\\\\n"
    latex_str += header
    latex_str += "\\midrule\n"
    
    # Find best and worst values for each column
    col_stats = {}
    for col in df.columns:
        valid_values = df[col].dropna()
        if len(valid_values) > 0:
            col_stats[col] = {
                'best': valid_values.max(),
                'worst': valid_values.min()
            }
    
    # Data rows with formatting
    for model, row in df.iterrows():
        model_clean = model.replace("_", "\\_").replace("-", "\\textminus{}")
        
        formatted_values = []
        for col, val in row.items():
            if pd.isna(val) or val is None:
                formatted_values.append("---")
            else:
                val_str = f"{val:.3f}"
                
                # Apply formatting based on best/worst
                if col in col_stats:
                    if abs(val - col_stats[col]['best']) < 1e-6:  # Best value
                        val_str = f"\\textbf{{{val_str}}}"
                    elif abs(val - col_stats[col]['worst']) < 1e-6:  # Worst value
                        val_str = f"\\textcolor{{red}}{{{val_str}}}"
                
                formatted_values.append(val_str)
        
        row_str = model_clean + " & " + " & ".join(formatted_values) + " \\\\\n"
        latex_str += row_str
    
    # End table
    latex_str += """\\bottomrule
\\end{tabular}
\\end{table}"""
    
    return latex_str

# Example usage for when you have whitespace data from another notebook:
def create_final_latex_table(jsonl_filename, whitespace_dict=None):
    """
    Create the final LaTeX table with optional whitespace data
    
    whitespace_dict format: {"model_name": whitespace_score, ...}
    """
    return load_jsonl_and_create_latex(
        jsonl_filename, 
        whitespace_data=whitespace_dict,
        caption="Multilingual Tokenization Performance Across Perturbation Categories",
        label="tab:multilingual_tokenization_performance"
    )

# Generate current LaTeX (without whitespace data)
current_latex = load_jsonl_and_create_latex(jsonl_file)
print("\n" + "="*50)
print("LaTeX Table String (with formatting):")
print("="*50)
print(current_latex)

print("\n" + "="*50)
print("Instructions for adding whitespace data:")
print("="*50)
print("""
1. Extract whitespace scores from your other notebook
2. Create a dictionary: whitespace_dict = {"model1": 0.85, "model2": 0.92, ...}
3. Use: final_latex = create_final_latex_table("tokenization_results.jsonl", whitespace_dict)
4. The table will highlight best values in bold and worst values in red
""")

whitespace_dict = { "Aya": 0.834685598377282, "BLOOM": 0.8194726166328601, "ByT5": 0.8062880324543611, "Comma": 0.845841784989858, "GPT-2": 0.8291925465838509, "GPT-4o": 0.8468559837728195, "Gemma-2": 0.8316430020283976, "Llama-3.2": 0.8194726166328601, "Phi-3": 0.8316430020283976, "Qwen-3": 0.8184584178498986, "Tekken": 0.8123732251521298, "TokenMonster": 0.7792746113989637, "XGLM": 0.9066937119675457, "mBERT": 0.901622718052738}
# final_latex, final_df = combine_results_and_create_final_table()
# print(final_latex)

LaTeX Table Data Structure:
Models: 14
Columns: ['Canonical', 'Writing Systems (Non-EN)', 'Noise (EN)', 'Noise (Non-EN)', 'Whitespace', 'LaTeX', 'Math/STEM (Non-EN)', 'Structural', 'Diacritics (Non-EN)']

Sample data:
Aya: {'Canonical': 1.0, 'Writing Systems (Non-EN)': 0.696, 'Noise (EN)': 0.799, 'Noise (Non-EN)': 0.673, 'Whitespace': 0.59, 'LaTeX': 0.611, 'Math/STEM (Non-EN)': 0.732, 'Structural': 0.452, 'Diacritics (Non-EN)': 0.485}
BLOOM: {'Canonical': 1.0, 'Writing Systems (Non-EN)': 0.662, 'Noise (EN)': 0.816, 'Noise (Non-EN)': 0.707, 'Whitespace': 0.557, 'LaTeX': 0.706, 'Math/STEM (Non-EN)': 0.603, 'Structural': 0.45, 'Diacritics (Non-EN)': 0.531}
ByT5: {'Canonical': 1.0, 'Writing Systems (Non-EN)': 0.671, 'Noise (EN)': 0.823, 'Noise (Non-EN)': 0.716, 'Whitespace': 0.568, 'LaTeX': 0.641, 'Math/STEM (Non-EN)': 0.717, 'Structural': 0.463, 'Diacritics (Non-EN)': 0.505}

DataFrame shape: (14, 9)

DataFrame preview:
       Canonical  Writing Systems (Non-EN)  Noise (EN)  Noise (Non-EN

In [680]:
import pandas as pd
import numpy as np
import json

def load_whitespace_from_json(filename="whitespace_results.json"):
    """Load whitespace results from JSON"""
    with open(filename, 'r') as f:
        return json.load(f)

# Extract whitespace data (run this in your other notebook)
# whitespace_data = extract_whitespace_data(all_summaries, "only_canonical_correct")
# save_whitespace_to_json(whitespace_data)

# Function to combine main results with whitespace data
def combine_results_and_create_final_table(main_jsonl="tokenization_results.jsonl", 
                                         whitespace_json="whitespace_results.json",
                                         caption="Multilingual Tokenization Performance Across Perturbation Categories",
                                         label="tab:multilingual_tokenization_performance"):
    """
    Combine main results with whitespace data and create final LaTeX table
    """
    
    # Load main results
    main_records = []
    with open(main_jsonl, 'r') as f:
        for line in f:
            main_records.append(json.loads(line))
    
    main_df = pd.DataFrame(main_records).set_index('model_name')
    
    # Load whitespace results
    try:
        with open(whitespace_json, 'r') as f:
            whitespace_data = json.load(f)
        
        # Add whitespace column
        main_df['Whitespace'] = main_df.index.map(whitespace_data)
        print("Whitespace data successfully added")
    except FileNotFoundError:
        print(f"Warning: {whitespace_json} not found. Creating table without whitespace data.")
    
    # Reorder columns to put Whitespace in the right position
    column_order = [
        "Writing Systems (Non-EN)", 
        "Noise (EN)",
        "Noise (Non-EN)",
        "Whitespace",  # Now properly positioned
        "LaTeX",
        "Math/STEM (Non-EN)",
        "NFD Affected", 
        "Diacritics (Non-EN)"
    ]
    
    # Only include columns that exist
    available_columns = [col for col in column_order if col in main_df.columns]
    main_df = main_df[available_columns]
    
    # Sort by average performance
    main_df['_avg_score'] = main_df.mean(axis=1, skipna=True)
    main_df = main_df.sort_values('_avg_score', ascending=False)
    main_df = main_df.drop('_avg_score', axis=1)
    
    # Generate LaTeX table with formatting
    latex_str = generate_formatted_latex_table(main_df, caption, label)
    
    return latex_str, main_df

def generate_formatted_latex_table(df, caption, label):
    """Generate compact LaTeX table matching the reference style with models as rows"""
    
    # Model name mapping for cleaner display
    model_mapping = {
        "google-byt5-small": "ByT5",
        "tokenmonster-englishcode-32000-consistent-v1": "TokenMonster", 
        "microsoft-Phi-3-mini-4k-instruct": "Phi-3",
        "gpt2": "GPT-2",
        "common-pile-comma-v0.1": "Comma",
        "google-bert-bert-base-multilingual-cased": "mBERT",
        "meta-llama-Llama-3.2-1B": "Llama-3.2",
        "mistralai-tekken": "Tekken",
        "Qwen-Qwen3-8B": "Qwen-3",
        "tiktoken-gpt-4o": "GPT-4o",
        "bigscience-bloom": "BLOOM",
        "aya-expanse": "Aya",
        "google-gemma-2-2b": "Gemma-2",
        "facebook-xglm-564m": "XGLM"
    }
    
    # Create new df with short names
    df_clean = df.copy()
    df_clean.index = [model_mapping.get(m, m) for m in df_clean.index]
    
    # Number of columns
    n_cols = len(df_clean.columns)
    
    # Start table with tabularx style
    latex_str = f"""\\begin{{table}}[hp]
\\centering
\\vspace*{{\\fill}}
\\footnotesize
\\begin{{tabularx}}{{\\textwidth}}{{l|*{{{n_cols}}}{{>{{\\centering\\arraybackslash}}X}}}}
\\toprule
\\textbf{{Model}} &"""
    
    # Compact column headers with rotation
    header_mapping = {
        "Writing Systems (Non-EN)": "WritSys",
        "Noise (EN)": "Noise\\textsubscript{EN}",
        "Noise (Non-EN)": "Noise\\textsubscript{NE}",
        "Whitespace": "Space",
        "LaTeX": "LaTeX",
        "Math/STEM (Non-EN)": "STEM\\textsubscript{NE}",
        "NFD Affected": "Unicode",
        "Diacritics (Non-EN)": "Diacr\\textsubscript{NE}"
    }
    
    for col in df_clean.columns:
        header_name = header_mapping.get(col, col)
        latex_str += f"\n\\rotatebox{{60}}{{\\textbf{{{header_name}}}}} &"
    latex_str = latex_str.rstrip(" &") + " \\\\\n\\midrule\n"
    
    # Find best values for each column (for bold formatting)
    best_values = {}
    for col in df_clean.columns:
        valid_vals = df_clean[col].dropna()
        if len(valid_vals) > 0:
            best_values[col] = valid_vals.max()
    
    # Add data rows
    for model in df_clean.index:
        latex_str += model
        for col in df_clean.columns:
            val = df_clean.loc[model, col]
            if pd.isna(val):
                latex_str += " & ---"
            else:
                val_str = f"{val:.2f}"
                # Bold the best value in each column
                if col in best_values and abs(val - best_values[col]) < 1e-6:
                    val_str = f"\\textbf{{{val_str}}}"
                latex_str += f" & {val_str}"
        latex_str += " \\\\\n"
    
    # Add summary statistics
    latex_str += "\\midrule\n"
    
    # Average row
    latex_str += "Avg"
    col_avgs = df_clean.mean(axis=0, skipna=True)
    best_avg = col_avgs.max() if len(col_avgs.dropna()) > 0 else None
    for col in df_clean.columns:
        avg_val = col_avgs[col]
        if pd.isna(avg_val):
            latex_str += " & ---"
        else:
            val_str = f"{avg_val:.2f}"
            # Bold the best average
            if best_avg is not None and abs(avg_val - best_avg) < 1e-6:
                val_str = f"\\textbf{{{val_str}}}"
            latex_str += f" & {val_str}"
    latex_str += " \\\\\n"
    
    # End table
    latex_str += f"""\\bottomrule
\\end{{tabularx}}
\\caption{{{caption}}}
\\label{{{label}}}
\\vspace*{{\\fill}}
\\end{{table}}"""
    
    return latex_str

# Example workflow:
print("WORKFLOW FOR COMBINING RESULTS:")
print("="*50)
print("""
1. In your OTHER notebook with whitespace data, run:
   whitespace_data = extract_whitespace_data(all_summaries, "only_canonical_correct")
   save_whitespace_to_json(whitespace_data)

2. In THIS notebook, run:
   final_latex, final_df = combine_results_and_create_final_table()
   print(final_latex)

3. The final LaTeX table will include:
   - All your current categories
   - Whitespace data from the other notebook
   - Bold formatting for best values
   - Red formatting for worst values
   - Proper LaTeX escaping
""")

WORKFLOW FOR COMBINING RESULTS:

1. In your OTHER notebook with whitespace data, run:
   whitespace_data = extract_whitespace_data(all_summaries, "only_canonical_correct")
   save_whitespace_to_json(whitespace_data)

2. In THIS notebook, run:
   final_latex, final_df = combine_results_and_create_final_table()
   print(final_latex)

3. The final LaTeX table will include:
   - All your current categories
   - Whitespace data from the other notebook
   - Bold formatting for best values
   - Red formatting for worst values
   - Proper LaTeX escaping



In [681]:

final_latex, final_df = combine_results_and_create_final_table()
print(final_latex)

Whitespace data successfully added
\begin{table}[hp]
\centering
\vspace*{\fill}
\footnotesize
\begin{tabularx}{\textwidth}{l|*{7}{>{\centering\arraybackslash}X}}
\toprule
\textbf{Model} &
\rotatebox{60}{\textbf{WritSys}} &
\rotatebox{60}{\textbf{Noise\textsubscript{EN}}} &
\rotatebox{60}{\textbf{Noise\textsubscript{NE}}} &
\rotatebox{60}{\textbf{Space}} &
\rotatebox{60}{\textbf{LaTeX}} &
\rotatebox{60}{\textbf{STEM\textsubscript{NE}}} &
\rotatebox{60}{\textbf{Diacr\textsubscript{NE}}} \\
\midrule
Phi-3 & 0.68 & 0.81 & 0.71 & 0.83 & 0.67 & \textbf{0.85} & 0.53 \\
mBERT & 0.62 & 0.78 & \textbf{0.73} & 0.90 & 0.63 & 0.75 & 0.53 \\
Tekken & 0.69 & 0.83 & 0.70 & 0.81 & 0.63 & 0.75 & 0.49 \\
XGLM & 0.65 & 0.84 & 0.72 & \textbf{0.91} & 0.63 & 0.63 & 0.51 \\
ByT5 & 0.67 & 0.82 & 0.72 & 0.81 & 0.64 & 0.72 & 0.51 \\
Qwen-3 & 0.68 & 0.82 & 0.70 & 0.82 & 0.59 & 0.71 & 0.53 \\
Comma & 0.68 & \textbf{0.87} & 0.71 & 0.85 & 0.61 & 0.64 & 0.49 \\
BLOOM & 0.66 & 0.82 & 0.71 & 0.82 & \textbf{0.71} & 0.60

## <a id='toc3_1_'></a>[Structural Text Elements](#toc0_)

In [656]:
styling_categories = [
    "Scripted text",
    "Macron Diacritic",
    "Fullwidth Characters",
    "Upside Down/Rotated",
    "Double struck",
    "Enclosed Characters",
    "Strikethrough",
    "Diacriticized styling",
    'Superscript/subscript',
    # "Subscript / Superscript"
]
style_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & all_summaries["task_pretty_name"].isin(styling_categories)
]
styled_df = get_styled_df(style_summaries, columns=["task_pretty_name", "langs"])
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Diacriticized styling,Double struck,Enclosed Characters,Fullwidth Characters,Macron Diacritic,Scripted text,Strikethrough,Superscript/subscript,Upside Down/Rotated,Mean
langs,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,
model_name,,,,,,,,,,
XGLM,0.387755,0.933333,0.777778,1.000000,0.275000,0.963636,0.295455,0.875000,0.142857,0.627868
GPT-2,0.489796,0.333333,0.361111,0.428571,0.435897,0.407407,0.386364,0.375000,0.500000,0.413053
Phi-3,0.489796,0.200000,0.277778,0.500000,0.300000,0.400000,0.477273,0.360000,0.571429,0.397364
Aya,0.425532,0.400000,0.305556,0.357143,0.250000,0.290909,0.409091,0.440000,0.642857,0.391232
Comma,0.468085,0.200000,0.277778,0.428571,0.358974,0.314815,0.522727,0.400000,0.428571,0.377725
ByT5,0.469388,0.400000,0.277778,0.500000,0.256410,0.388889,0.409091,0.400000,0.285714,0.376363
GPT-4o,0.442308,0.294118,0.333333,0.375000,0.375000,0.368421,0.479167,0.333333,0.375000,0.375076
Llama-3.2,0.428571,0.312500,0.250000,0.400000,0.350000,0.303571,0.477273,0.360000,0.466667,0.372065


In [505]:
save_path = OUTPUT_DIR / "style_summary.tsv"

style_summaries.to_csv(save_path, sep="\t")
print(f"File saved at \n{save_path}")

File saved at 
output/results-v5/summary/all/style_summary.tsv


In [506]:
styled_df = get_styled_df(style_summaries, columns=["task_pretty_name", "langs"])

# Set caption
styled_df = styled_df.set_caption(
    f"Unicode Characters (Green=Best, Red=Worst) {FILTER}"
)
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Diacriticized styling,Double struck,Enclosed Characters,Fullwidth Characters,Macron Diacritic,Scripted text,Strikethrough,Upside Down/Rotated,Mean
langs,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,eng_Latn,
model_name,,,,,,,,,
XGLM,0.387755,0.933333,0.777778,1.000000,0.275000,0.963636,0.295455,0.142857,0.596977
GPT-2,0.489796,0.333333,0.361111,0.428571,0.435897,0.407407,0.386364,0.500000,0.417810
Phi-3,0.489796,0.200000,0.277778,0.500000,0.300000,0.400000,0.477273,0.571429,0.402034
Aya,0.425532,0.400000,0.305556,0.357143,0.250000,0.290909,0.409091,0.642857,0.385136
GPT-4o,0.442308,0.294118,0.333333,0.375000,0.375000,0.368421,0.479167,0.375000,0.380293
Comma,0.468085,0.200000,0.277778,0.428571,0.358974,0.314815,0.522727,0.428571,0.374940
Llama-3.2,0.428571,0.312500,0.250000,0.400000,0.350000,0.303571,0.477273,0.466667,0.373573
ByT5,0.469388,0.400000,0.277778,0.500000,0.256410,0.388889,0.409091,0.285714,0.373409


In [507]:
multilingual_noise = [
    "Keyboard proximity errors",
    # "Space removal",
    # "Character deletion",
    # "Typographical errors",
    # "OCR Errors",
]

FILTER = "only_canonical_correct"
multilingual_noise_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(multilingual_noise))
    # & (~all_summaries["task"].str.contains("stem"))
]

styled_df = get_styled_df(multilingual_noise_summaries)
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


## <a id='toc3_2_'></a>[Math](#toc0_)

In [508]:
math_perturbs = [
    "Spelled out",
    "Superscript/subscript",
    "LaTeX",
    "Decorative Unicode",
    "Farsi",
    "Turkish",
    "Italian",
    "Chinese",
]

FILTER = "only_canonical_correct"
math_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(math_perturbs))
    & (all_summaries["task"].str.contains("math"))
]

styled_df = get_styled_df(math_summaries, ["task_pretty_name"])
styled_df = styled_df.set_caption(f"MATH {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Chinese,Decorative Unicode,Farsi,Italian,LaTeX,Spelled out,Turkish,Mean
model_name,,,,,,,,
Phi-3,0.764706,0.470588,0.705882,1.000000,0.882353,0.294118,0.941176,0.722689
mBERT,0.857143,0.428571,0.642857,0.785714,0.928571,0.357143,0.714286,0.673469
ByT5,0.733333,0.533333,0.866667,0.600000,0.866667,0.333333,0.666667,0.657143
GPT-4o,0.842105,0.421053,0.578947,0.894737,0.789474,0.368421,0.684211,0.654135
Tekken,0.647059,0.529412,0.764706,0.882353,0.647059,0.352941,0.705882,0.647059
Aya,0.642857,0.357143,0.714286,0.714286,0.785714,0.428571,0.857143,0.642857
TokenMonster,0.785714,0.571429,0.500000,0.714286,0.785714,0.428571,0.714286,0.642857
Llama-3.2,0.764706,0.352941,0.647059,0.882353,0.882353,0.352941,0.470588,0.621849
XGLM,0.733333,0.733333,0.533333,0.666667,0.733333,0.333333,0.600000,0.619048


In [509]:
latex_perturbs = [
    # "Spelled out",
    # "Superscript/subscript",
    "LaTeX",
    # "Decorative Unicode",
    # "Farsi",
    # "Turkish",
    # "Italian",
    # "Chinese",
]

FILTER = "only_canonical_correct"
math_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(latex_perturbs))
    # & (all_summaries["task"].str.contains("math"))
]

styled_df = get_styled_df(math_summaries, ["task_pretty_name", "group_name"])
styled_df = styled_df.set_caption(f"LaTeX {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


# <a id='toc4_'></a>[Categorical Groups](#toc0_)

## <a id='toc4_1_'></a>[Morphological](#toc0_)

In [510]:
morph_perturbs = [
    "Contractions",
        "Inflections",
        "Compounds",
        "Derivations",
        "Morpheme separation",
        # "Spelled out",
]

FILTER = "only_canonical_correct"
morph_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(morph_perturbs))
    # & (all_summaries["task"].str.contains("math"))
]

styled_df = get_styled_df(morph_summaries, ["task_pretty_name", "langs"])
styled_df = styled_df.set_caption(f"Morphological Challenges, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


## <a id='toc4_2_'></a>[LAnguage Contact](#toc0_)

In [511]:
contact_perturbs = [
    "Romanization",
    "Code/language/script switching",
    "Historical spelling"
]

FILTER = "only_canonical_correct"
contact_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(contact_perturbs))
    # & (all_summaries["task"].str.contains("math"))
]

styled_df = get_styled_df(contact_summaries, ["task_pretty_name", "langs"])
styled_df = styled_df.set_caption(f"Language Contact, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [512]:
contact_perturbs = [
    "Similar words",
    "Equivalent expressions"
]

## TODO: again with vocab size

FILTER = "only_canonical_correct"
contact_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(contact_perturbs))
    # & (all_summaries["task"].str.contains("math"))
]
contact_summaries["vocab_bucket"] = contact_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(contact_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"Morphological Challenges, {FILTER}")
styled_df


/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/149573599.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  contact_summaries["vocab_bucket"] = contact_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [513]:

styled_df = get_styled_df(contact_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"Morphological Challenges, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


# <a id='toc5_'></a>[Orthography](#toc0_)

In [516]:
all_summaries["task_pretty_name"].unique()

array(['Canonical', 'LaTeX', 'Spelled out', 'Decorative Unicode', 'Farsi',
       'Turkish', 'Italian', 'Chinese', 'Space removal',
       'Orthographic errors', 'Plausible diacritics errors',
       'English keyboard', 'Typographical errors', 'Grammatical errors',
       'Contractions', 'Code/language/script switching', 'Capitalization',
       'Optional diacritics', 'Colloquial', 'Dialects', 'Compounds',
       'Inflections', 'Emoji substitution', 'Abbreviations', 'OCR Errors',
       'Keyboard proximity errors',
       'Word Spacing/Zero-width characters/Extra Space',
       'Word reordering', 'Romanization', 'Traditional', 'Derivations',
       'Equivalent expressions', 'Character deletion',
       'Partially romanized', 'Arabic Keyboard for Farsi',
       'Historical spelling', 'Homoglyphs', 'Hyphenated spelling',
       'Similar words', 'Letter repetition for emphasis',
       'Web search query', 'Lowercase', 'Macron Diacritic',
       'Scripted text', 'Spaced styling', 'Superscr

In [517]:
perturbs = [
    "Code/language/script switching",
    "Traditional",
    "Romanization",
    "Partially romanized"
]
CAPTION = "Writing System Variations"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3557481691.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [552]:
perturbs = [
    # "Code/language/script switching",
    # "Traditional",
    # "Romanization",
    # "Partially romanized"
    "Orthographic errors"
]
CAPTION = "Orthographic Errors"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2058586945.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


## <a id='toc5_1_'></a>[Input Medium](#toc0_)

In [518]:
## input medium

perturbs = [
    "English keyboard",
    "Arabic Keyboard for Farsi",
    "Word Spacing/Zero-width characters/Extra Space",
    "Homoglyphs"
]
CAPTION = "Input Medium Challenges"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3730214633.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [522]:
## input medium

perturbs = [
    "English keyboard",
    # "Arabic Keyboard for Farsi",
    # "Word Spacing/Zero-width characters/Extra Space",
    # "Homoglyphs"
]
CAPTION = "Input Medium Differences"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2588012899.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [ ]:
## input medium noise

perturbs = [
    # "English keyboard",
    # "Arabic Keyboard for Farsi",
    "Word Spacing/Zero-width characters/Extra Space",
    "Homoglyphs"
]
CAPTION = "Input Medium Noise"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]

## TODO: group by category of nromalization
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/4141387247.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


## <a id='toc5_2_'></a>[Diacritics](#toc0_)

In [554]:
perturbs = [
    "Optional diacritics",
    "Romanization"
    # "Plausible diacritics errors",
    # "Macron Diacritic",
    # "Diacriticized styling"
]
CAPTION = "Diacritics"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/49966288.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


## <a id='toc5_3_'></a>[Register Style](#toc0_)

In [496]:
perturbs = [
    "Web search query",
    "Abbreviations",
    "Word reordering",
    "Historical spelling",
    "Colloquial",
    "Emoji substitution",
    "Character substitution",
    "Letter repetition for emphasis"
]
CAPTION = "Register & Style"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3402975788.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3876124597.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [497]:
perturbs = [
    "Web search query",
    "Abbreviations",
    "Word reordering",
    # "Historical spelling",
    # "Colloquial",
    # "Emoji substitution",
    # "Character substitution",
    # "Letter repetition for emphasis"
]
CAPTION = "Register & Style"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3581615980.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3876124597.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [499]:
perturbs = [
    # "Web search query",
    # "Abbreviations",
    # "Word reordering",
    # "Historical spelling",
    "Colloquial",
    "Emoji substitution",
    "Character substitution",
    "Letter repetition for emphasis"
]
CAPTION = "Register & Style"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3098911458.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3876124597.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


## <a id='toc5_4_'></a>[Morph](#toc0_)

In [523]:
perturbs = [
    "Contractions",
    "Compounds",
    "Inflections",
    "Derivations",
    "Morpheme separation"
]
CAPTION = "Morphological Challenges"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2332631894.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [ ]:
perturbs = [
    "Contractions",
    # "Compounds",
    # "Inflections",
    # "Derivations",
    # "Morpheme separation"
]
CAPTION = "Morphological Challenges"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

# The sky is
# The sky's
# The number of legs of a cow has is
# The number of legs a cow's got is
# Dr Smith is doctor
# Dr Smith's a doctor

# dell'
# bloom -> don't shan't-> 2 tokens

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3378400938.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [527]:
perturbs = [
    # "Contractions",
    "Compounds",
    # "Inflections",
    # "Derivations",
    # "Morpheme separation"
]
CAPTION = "Morphological Challenges"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["vocab_bucket"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3894085822.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Compounds,Mean
langs,eng_Latn,
vocab_bucket,,
Medium,0.837209,0.837209
Large,0.832558,0.832558
Small,0.822485,0.822485
X-Small,0.785714,0.785714
────────────────────,─────,─────
MEAN,0.819492,0.819492
MAX,0.837209,0.837209
MIN,0.785714,0.785714


In [530]:
perturbs = [
    # "Contractions",
    # "Compounds",
    # "Inflections",
    "Derivations",
    # "Morpheme separation"
]
CAPTION = "Morphological Challenges"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2447158707.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Derivations,Mean
langs,tur_Latn,
model_name,,
Comma,0.850000,0.850000
Aya,0.829268,0.829268
GPT-4o,0.829268,0.829268
Llama-3.2,0.829268,0.829268
Qwen-3,0.829268,0.829268
Gemma-2,0.804878,0.804878
Tekken,0.804878,0.804878
GPT-2,0.800000,0.800000


# <a id='toc6_'></a>[Noise Cuated](#toc0_)

In [531]:
perturbs = [
    "OCR Errors",
    "Character deletion",
    "Space removal",
    "Typographical errors"
]
CAPTION = "Noise"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1602138391.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


# <a id='toc7_'></a>[Grammar](#toc0_)

In [532]:
perturbs = [
    "Grammatical errors"
]
CAPTION = "Grammatical Errors"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/852252785.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


# <a id='toc8_'></a>[Linguistic Variety](#toc0_)

In [557]:
perturbs = [
    # "Equivalent expressions",
    # "Similar words",
    "Dialects",
    "Colloquial"
]
CAPTION = "Linguistic Variety"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3839392992.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [558]:
perturbs = [
    "Equivalent expressions",
    "Similar words",
    # "Dialects"
]
CAPTION = "Linguistic Variety"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
# styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["vocab_bucket"])
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/3290161057.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


# <a id='toc9_'></a>[Structural Text](#toc0_)

In [534]:
perturbs = [
    "Decorative Unicode",
    "LaTeX",
    "Spelled out",
    "Scripted text",
    "Double struck",
    "Enclosed Characters",
    "Fullwidth Characters",
    "Strikethrough",
    "Upside Down/Rotated",
    "Spaced styling",
    "Hyphenated spelling",
    "Capitalization",
    "Lowercase",
    "Superscript/subscript styling",
    "Superscript/subscript",
    "Unusual formatting",
    "Unicode formatting"
]
CAPTION = "Structural Text Elements"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2078125851.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [546]:
perturbs = [
    "Spaced styling",
    "Hyphenated spelling",
    "Capitalization",
    "Lowercase",
]
CAPTION = "Structural Text Elements"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
    #  italian capitalizations are once at a time
    & (all_summaries["langs"] != "ita_Latn")
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1910321383.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Capitalization,Hyphenated spelling,Lowercase,Spaced styling,Mean
langs,eng_Latn,eng_Latn,eng_Latn,eng_Latn,
model_name,,,,,
Aya,0.825000,0.325000,0.894737,0.450000,0.623684
Tekken,0.800000,0.375000,0.921053,0.350000,0.611513
BLOOM,0.875000,0.350000,0.921053,0.275000,0.605263
Gemma-2,0.825000,0.325000,0.921053,0.350000,0.605263
Llama-3.2,0.950000,0.250000,0.894737,0.325000,0.604934
Qwen-3,0.850000,0.225000,0.868421,0.450000,0.598355
TokenMonster,0.923077,0.282051,0.891892,0.282051,0.594768
GPT-2,0.743590,0.358974,0.891892,0.358974,0.588358


In [544]:
perturbs = [
    "Unusual formatting",
    "Unicode formatting"
]
CAPTION = "General Unicode Formatting"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/4283084656.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Unicode formatting,Unusual formatting,Mean
langs,eng_Latn,eng_Latn,
model_name,,,
BLOOM,0.894737,0.800000,0.847368
TokenMonster,0.909091,0.733333,0.821212
Llama-3.2,0.916667,0.666667,0.791667
Phi-3,0.875000,0.687500,0.781250
Comma,0.909091,0.625000,0.767045
mBERT,0.826087,0.687500,0.756793
Gemma-2,0.900000,0.600000,0.750000
Qwen-3,0.863636,0.625000,0.744318


### <a id='toc9_1_1_'></a>[Math styling](#toc0_)

In [540]:
perturbs = [
    "Superscript/subscript",
    "Double struck"
]
CAPTION = "Unicode Mathematical/Scientific Styling"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/2405335638.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


task_pretty_name,Double struck,Superscript/subscript,Superscript/subscript styling,Mean
langs,eng_Latn,eng_Latn,eng_Latn,
model_name,,,,
XGLM,0.933333,0.875000,0.925000,0.911111
Tekken,0.375000,0.370370,0.350000,0.365123
Aya,0.400000,0.440000,0.225000,0.355000
ByT5,0.400000,0.400000,0.256410,0.352137
Gemma-2,0.375000,0.370370,0.300000,0.348457
Qwen-3,0.333333,0.360000,0.325000,0.339444
GPT-2,0.333333,0.375000,0.307692,0.338675
BLOOM,0.333333,0.407407,0.275000,0.338580


In [538]:
perturbs = [
    "Numerical formats",
    "Number Romanization",
    "Date formats",
    # "Spelled out",
]
CAPTION = "Numerical & Format Variations"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/421030021.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)


In [539]:
perturbs = [
    # "Numerical formats",
    # "Number Romanization",
    # "Date formats",
    "Spelled out",
]
CAPTION = "Numerical & Format Variations"
FILTER = "only_canonical_correct"
distilled_summaries = all_summaries[
    (all_summaries["filtering_mode"] == FILTER)
    & (all_summaries["task_pretty_name"].isin(perturbs))
]
distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
styled_df = get_styled_df(distilled_summaries, ["task_pretty_name", "langs"], index=["model_name"])
styled_df = styled_df.set_caption(f"{CAPTION}, {FILTER}")
styled_df

/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1299645984.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  distilled_summaries["vocab_bucket"] = distilled_summaries["model_name"].map(VOCAB_BUCKETS_PRETTY_NAME)
/var/folders/8n/l2sy_xnn4fv5j3b8z0dhfxlh0000gn/T/ipykernel_35969/1679956316.py:97: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(weighted_avg)
